## Widgets

In [0]:
%python
dbutils.widgets.removeAll()

#remove bidgets

In [0]:
%sql
--- crear widgets

create widget text storageName default "adlssmartprojectdev13";
create widget text catalogo default "project_sedev";
create widget text container default "semiestrcturprojectdev";
create widget text credential_ default "credential_dev";

In [0]:
storageName = dbutils.widgets.get("storageName")
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
credential_ = dbutils.widgets.get("credential_")


# guardar el widget en una variable

In [0]:
path = str("abfss://") + str(dbutils.widgets.get("container")) + str("@") + str(dbutils.widgets.get("storageName")) + str(".dfs.core.windows.net/")

##Catalogs y Schemas

In [0]:
%sql
--- crear el catalogo

CREATE CATALOG IF NOT EXISTS ${catalogo};

In [0]:
%sql
---  crear los eschemas y volume necesario

CREATE SCHEMA IF NOT EXISTS ${catalogo}.raw;
CREATE SCHEMA IF NOT EXISTS ${catalogo}.bronze;
CREATE SCHEMA IF NOT EXISTS ${catalogo}.silver;
CREATE SCHEMA IF NOT EXISTS ${catalogo}.gold;

CREATE VOLUME IF NOT EXISTS ${catalogo}.raw.datasets;

## external locations

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-raw-data_se`
URL 'abfss://${container}@${storageName}.dfs.core.windows.net/raw/'
WITH (STORAGE CREDENTIAL ${credential_})
COMMENT 'Ubicación externa para las tablas raw del Data Lake en la carpeta raw_data_project';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-bronze_se`
URL 'abfss://${container}@${storageName}.dfs.core.windows.net/bronze/'
WITH (STORAGE CREDENTIAL ${credential_})
COMMENT 'Ubicación externa para las tablas bronze del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-silver_se`
URL 'abfss://${container}@${storageName}.dfs.core.windows.net/silver/'
WITH (STORAGE CREDENTIAL ${credential_})
COMMENT 'Ubicación externa para las tablas silver del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-gold_se`
URL 'abfss://${container}@${storageName}.dfs.core.windows.net/gold/'
WITH (STORAGE CREDENTIAL ${credential_})
COMMENT 'Ubicación externa para las tablas gold del Data Lake';

In [0]:
%sql
--#eliminar tabla en catalogo
DROP TABLE IF EXISTS ${catalogo}.bronze.instances_train;
DROP TABLE IF EXISTS ${catalogo}.silver.annotations_polygon_clean;
DROP TABLE IF EXISTS ${catalogo}.silver.annotations_rle_raw ;

In [0]:
## eliminar files en el storage
dbutils.fs.rm(f"{path}/bronze/instance_train",True)
dbutils.fs.rm(f"{path}/silver/annotations_polygon_clean",True)
dbutils.fs.rm(f"{path}/silver/annotations_rle_raw",True)

#TABLAS BRONZE

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ${catalogo}.bronze.instance_train (
)
USING DELTA
LOCATION "abfss://${container}@${storageName}.dfs.core.windows.net/bronze/instance_train"

## tablas silver

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ${catalogo}.silver.annotations_polygon_clean (
)
USING DELTA

LOCATION "abfss://${container}@${storageName}.dfs.core.windows.net/silver/annotations_polygon_clean"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ${catalogo}.silver.annotations_rle_raw (
product_id string,
product_category_name string, 
product_category_name_english string 
)
USING DELTA

LOCATION "abfss://${container}@${storageName}.dfs.core.windows.net/silver/annotations_rle_raw"

## tabla gold